*Title:* 5_Manhattan Plots
*Goal:* Prepare data for manhatten plots and the graph.

# 5.1 Prepare data
*Goal:* Prepare the file for plotting (add cutoff values, looping for 2016 and 2017 files)
Based off of data-manipulation-fst-output-file

``` R
module load r/4.4.0 
Rscript 3.e.ii.add_cutoff.R
``` 

RScript:
``` R
library(tidyverse)
library(data.table)

process_file <- function(input_file_name, output_file_name) {
  # Read the input file
  df <- read.table(input_file_name, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
  
  # Reshape the data from wide to long format
  tidy_data <- df %>%
    pivot_longer(cols = 2:7, names_to = "population", values_to = "Fst_poolfstat") %>%
    separate(Chromosome_Position, into = c("Chromosome", "Position"), sep = "_") %>% # splitting Chromosome and Position columns
    mutate(Chromosome = recode(Chromosome,
      "chrI" = 1, "chrII" = 2, "chrIII" = 3, "chrIV" = 4, "chrV" = 5,
      "chrVI" = 6, "chrVII" = 7, "chrVIII" = 8, "chrIX" = 9, "chrX" = 10,
      "chrXI" = 11, "chrXII" = 12, "chrXIII" = 13, "chrXIV" = 14, "chrXV" = 15,
      "chrXVI" = 16, "chrXVII" = 17, "chrXVIII" = 18, "chrXIX" = 19, "chrXX" = 20,
      "chrXXI" = 21
    )) %>% # converting Chromosome values to digits
    mutate(Cutoff = NA) # initialize Cutoff column with NA

  # Calculate outliers
  calculate_outliers <- function(tidy_data) {
    result_list <- list()
    unique_samples <- unique(tidy_data$population)
    
    for (sample in unique_samples) {
      subset_data <- tidy_data[tidy_data$population == sample, ]
      sub_Fst_poolfstat <- subset_data$Fst_poolfstat
      threshold <- quantile(sub_Fst_poolfstat, probs = 0.95, na.rm = TRUE) # 95th percentile threshold
      outliers <- which(sub_Fst_poolfstat > threshold)  # identifies outliers
      min_outlier <- min(sub_Fst_poolfstat[outliers], na.rm = TRUE)
      max_outlier <- max(sub_Fst_poolfstat[outliers], na.rm = TRUE)
      
      result_list[[sample]] <- list(CutoffValue = threshold, MinOutlier = min_outlier, MaxOutlier = max_outlier)
    }
    return(result_list)
  }
  
  results <- calculate_outliers(tidy_data)
  print(results)  # Print out the results for debugging

  # Add cutoff values to the data
  for (pop_name in names(results)) {
    match_rows <- tidy_data$population == pop_name
    tidy_data$Cutoff[match_rows] <- results[[pop_name]]$CutoffValue
  }

  # Write the processed data to the output file
  write.table(tidy_data, output_file_name, sep = "\t", col.names = TRUE, row.names = FALSE, quote = FALSE)
}

# Define input and output file names
#Making a loop for both 2016 abd 2017 to run at same time :)
input_file_1 <- "3.d.ii2017_poolfstat_SNP_FST_with_locations.tsv"
output_file_1 <- "3.e.ii2017_poolfstat_SNP_FST_with_locations_cutoff.tsv"

input_file_2 <- "3.d.ii2016_poolfstat_SNP_FST_with_locations.tsv"  
output_file_2 <- "3.e.ii2016_poolfstat_SNP_FST_with_locations_cutoff.tsv"  

# Process both files
process_file(input_file_1, output_file_1)
process_file(input_file_2, output_file_2)
``` 